# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata
metadata_json = dataset.metadata.to_json()
print('Dataset name:', metadata_json.get('name', 'N/A'))
print('Description:', metadata_json.get('description', 'N/A'))

## 2. Data Overview
Review available record sets and their IDs, fields, and columns.

Entities are referenced by their `@id` as defined by the Croissant schema.

In [ ]:
# List all record sets with their @id and available fields
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"- Record Set: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - Field: {field.id} (name: {field.name}, dataType: {getattr(field, 'data_type', None)})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the overview above.

Here, we extract all record sets for inspection, referencing exclusively by their `@id`.

In [ ]:
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for Record Set {record_set_id}:")
    print(df.columns.tolist())
    print()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. All references use the entity `@id` values.

_Note: Please refer to the output above for record set and field `@id` values. The following code demonstrates EDA on a common field, such as a numeric abundance, peptide count, or molecular weight field, as appropriate for mass spectrometry tables._

In [ ]:
# Example: Choose a record set and numeric field for EDA.

# For demonstration, select the first record set and try to find a numeric field
example_record_set_id = record_set_ids[0]
df = dataframes[example_record_set_id]

# Try to automatically detect a likely numeric field
numeric_candidates = []
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_candidates.append(col)

# If no numeric columns detected, try to convert common quantitative columns
if not numeric_candidates:
    # Attempt to auto-convert columns to numeric (ignore errors)
    for col in df.columns:
        df[col + "_as_num"] = pd.to_numeric(df[col], errors="coerce")
        if df[col + "_as_num"].notna().sum() > 0:
            numeric_candidates.append(col + "_as_num")
if len(numeric_candidates) == 0:
    print("No numeric fields found for EDA.")
else:
    numeric_field_id = numeric_candidates[0]
    print(f"Using numeric field '{numeric_field_id}' in record set '{example_record_set_id}'.")

    # Filter based on a threshold
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'fi' else 10
    filtered_df = df[df[numeric_field_id] > threshold]

    print(f"Filtered records in '{example_record_set_id}' with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by another common column, e.g., a categorical column
    group_candidates = [c for c in df.columns if c != numeric_field_id and df[c].nunique() < (len(df) // 10) and pd.api.types.is_object_dtype(df[c])]
    if group_candidates:
        group_field = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index().sort_values(by=numeric_field_id, ascending=False)
        print(f"\nGrouped data by '{group_field}' (mean of '{numeric_field_id}'):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(numeric_candidates) > 0:
    # Plot histogram of the numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in Record Set '{example_record_set_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If a grouping was found, visualize top groups
    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 6))
        topk = min(15, len(grouped_df))
        sns.barplot(data=grouped_df.head(topk), x=numeric_field_id, y=group_field, palette='viridis')
        plt.title(f"Top {topk} '{group_field}' by mean '{numeric_field_id}'")
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, inspect, and analyze a FAIR-compliant mass spectrometry dataset using the `mlcroissant` library by referencing all entities using their Croissant `@id`. The approach ensures reproducibility and semantic clarity in data workflows, suitable for both prospective data standardization and downstream machine learning tasks.

**Key findings:**
- Dataset was loaded and basic metadata reviewed.
- All data entity references were made via Croissant `@id`.
- Tabular data from each record set was loaded, filtered, normalized, grouped, and visualized.

_Further domain analysis (e.g., specific protein scores or biological inferences) would require consulting the full record set and schema documentation._